## **Step 1: Define the tools**

In [22]:
from langchain_openai import ChatOpenAI
import os 
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model="gpt-5-mini")

In [23]:
from langchain.tools import tool

In [ ]:
@tool
def tool_duckduckgo_search(query: str) -> str:
    """Use this tool to gather information about current events or general knowledge"""

    from langchain_community.tools import DuckDuckGoSearchRun
    search = DuckDuckGoSearchRun()
    
    response = search.invoke(query)
    return response

    
tool_duckduckgo_search.invoke("What is the capital of France?")


In [ ]:
@tool
def tool_wikipedia_search(query: str) -> str:
    """Use this tool to gather information about historical events"""

    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper
    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

    response = wikipedia.invoke(query)
    return response
    
tool_wikipedia_search.invoke("Alan Turing")


In [ ]:
@tool
def tool_arxiv_search(query: str) -> str:
    """Use this tool to gather information about arXiv papers"""

    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    #1. Initialize the ArxivAPIWrapper
    arxiv_api_wrapper = ArxivAPIWrapper(
        top_k_results=3,
        doc_content_chars_max=4000
    )
    
    #2. Initialize the Arxiv Query Object
    arxiv = ArxivQueryRun(api_wrapper=arxiv_api_wrapper)

    #3. Invoke the Arxiv Query Object
    response = arxiv.invoke(query)
    return response

tool_arxiv_search.invoke("What are the latest papers on AI?")

In [38]:
@tool
def personal_info(query: str) -> str:
    """Use this tool when you need to answer questions about personal information"""
    
    infos = [
        {
            "name": "Ojas Dighe",
            "age": 25,
            "location": "India",
            "interests": "coding, building things, reading, writing"
        },
        {
            "name": "John Doe",
            "age": 30,
            "location": "USA",
            "interests": "coding, building things, reading, writing"
        },
        {
            "name": "Jane Smith",
            "age": 28,
            "location": "Canada",
            "interests": "coding, building things, reading, writing"
        }
    ]

    for info in infos:
        if info["name"] == query:
            return f"Name: {info['name']}, Age: {info['age']}, Location: {info['location']}, Interests: {info['interests']}"


personal_info.invoke("Ojas Dighe")

'Name: Ojas Dighe, Age: 25, Location: India, Interests: coding, building things, reading, writing'

## Bind Tools

In [ ]:
toolkit = [tool_duckduckgo_search, tool_wikipedia_search, tool_arxiv_search, personal_info]

# Step 1: Tool Binding
llm_bind = llm.bind_tools(toolkit)

llm_bind.invoke("What is the capital of France?")

AIMessage(content='The capital of France is Paris.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 223, 'total_tokens': 303, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUAzV1BGqyhMdsq0hLAuvsQGcIdkN', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d86e2-2495-7541-bd8f-23009a26408f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 223, 'output_tokens': 80, 'total_tokens': 303, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 64}})

In [ ]:
# We will not be able to get content in LLM response even after tool binding. 
# You will see tool calls in the response, but no content in AIMessage Object.
# To solve this, we need to use a tool calling agent. Which is created in ReAct_Agent.ipynb
llm_bind.invoke("Tell me about Ojas Dighe. Make tool calls if necessary")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 229, 'total_tokens': 259, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUAzXXIxvnCwbkQcNjCnFoMaQlUDa', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d86e2-2f78-7401-a25c-84a8467fdae7-0', tool_calls=[{'name': 'tool_duckduckgo_search', 'args': {'query': 'Ojas Dighe'}, 'id': 'call_1qXdq6APOMxsZjgoySZzMwjs', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 229, 'output_tokens': 30, 'total_tokens': 259, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})